In [3]:
import pandas as pd
import numpy as np
import pandas_ta as ta
import yfinance as yf
import requests # requests sert à envoyer une requête vers un site web pour récupérer son contenu.
from io import StringIO

In [4]:
import sys
print(sys.executable)


C:\Users\bilal\Desktop\ProjetPython\FinAgent-X An Explainable Multi-Agent Financial Intelligence Platform\.venv\Scripts\python.exe


In [5]:
end_date = pd.to_datetime('2023-01-01')
start_date = pd.to_datetime(end_date) - pd.DateOffset(years=5)
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"

In [6]:
def get_sp500_tickers(url):
    """
    Get the list of S&P 500 tickers from Wikipedia.
    """
    headers = {
        "User-Agent": "Mozilla/5.0"
    }
    response = requests.get(url, headers=headers)
    if response.status_code == 200:
        sp500_table = pd.read_html(StringIO(response.text))[0]
    return sp500_table['Symbol'].str.replace(".", "-").tolist()    

def download_data(tickers, start_date, end_date):
    """
    Download historical stock data for the given tickers and date range.
    """
    df = yf.download(tickers, start=start_date, end=end_date, group_by='ticker', auto_adjust=True, threads=True)
    return df



Imagine qu’une action fasse un split 2 pour 1 :

avant le split, une action valait 100

après le split, elle vaut 50

Si tu regardes les prix bruts, tu vois une grosse baisse soudaine, même si la valeur de l’entreprise n’a pas réellement changé.

Avec auto_adjust=True, pandas/yfinance réécrit les prix pour que cette baisse soit “corrigée” :

le prix avant split devient 100

le prix après split devient 50, mais le cours est ajusté pour garder une série continue
ainsi, l’historique reste plus cohérent pour l’analyse

Exemple concret :

Sans ajustement :

2024-01-01 : 100

2024-01-02 : 50

Avec auto_adjust=True :

le logiciel “recalcule” la série pour que l’évolution soit plus logique et plus comparable dans le temps

Pourquoi c’est utile :

éviter les fausses baisses dues aux splits

avoir une courbe de prix plus stable pour faire des graphiques ou des analyses
En bref :

auto_adjust=True = “corrige les prix pour tenir compte des événements comme les splits”

Un header, c’est une information envoyée avec la requête HTTP.

Ici, on dit au site :

Je suis un navigateur web normal

Sans ça, Wikipedia peut répondre :

403 Forbidden

Parce qu’il pense que la requête vient d’un robot ou d’un script automatique.<br>
Les statuts importants :<br>

<Response [200]>  = OK, page récupérée<br>
<Response [403]>  = Forbidden, accès bloqué<br>
<Response [404]>  = page introuvable<br>
<Response [500]>  = erreur serveur<br>

Tu utilises StringIO quand tu as déjà du texte en mémoire, mais qu’une fonction veut lire ce texte comme si c’était un fichier.

In [ ]:
tickers = get_sp500_tickers(url)
data = download_data(tickers, start_date, end_date)

[**********            20%                       ]  103 of 503 completed$GEV: possibly delisted; no price data found  (1d 2018-01-01 00:00:00 -> 2023-01-01 00:00:00) (Yahoo error = "Data doesn't exist for startDate = 1514782800, endDate = 1672549200")
[***********           23%                       ]  116 of 503 completed$FDXF: possibly delisted; no price data found  (1d 2018-01-01 00:00:00 -> 2023-01-01 00:00:00) (Yahoo error = "Data doesn't exist for startDate = 1514782800, endDate = 1672549200")
[*****************     36%                       ]  182 of 503 completed$VLTO: possibly delisted; no price data found  (1d 2018-01-01 00:00:00 -> 2023-01-01 00:00:00) (Yahoo error = "Data doesn't exist for startDate = 1514782800, endDate = 1672549200")
[**********************57%**                     ]  286 of 503 completed$SNDK: possibly delisted; no price data found  (1d 2018-01-01 00:00:00 -> 2023-01-01 00:00:00) (Yahoo error = "Data doesn't exist for startDate = 1514782800, endDate = 16

In [ ]:
data = data.stack()
data.columns.name = None
data.reset_index(inplace=True)
data.columns = [c.lower().strip() for c in data.columns]
data.head()


C:\Users\bilal\AppData\Local\Temp\ipykernel_26016\794733328.py:1: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  df = df.stack(level=0)


## Définition

La méthode `stack()` **déplace un niveau des colonnes vers l'index**.

- Si les colonnes sont simples (`Index`), elles deviennent un **deuxième niveau de l'index**.
- Si les colonnes sont un `MultiIndex`, `stack()` déplace **le dernier niveau** des colonnes vers l'index (par défaut).

---

## Syntaxe

```python
df.stack(level=-1)
```

- `level=-1` (par défaut) : empile le dernier niveau des colonnes.
- `level=0` ou `level='nom_du_niveau'` : empile un niveau spécifique.

---

In [ ]:
data.info()
display(data.describe())
print(data.isnull().sum())
data.drop_duplicates(subset=["date", "ticker"],inplace=True)
data.head(10)

In [ ]:
print(data.isnull().sum())
print(data.index.names)

Price
open      0
high      0
low       0
close     0
volume    0
dtype: int64
['Date', 'Ticker']


In [ ]:
data = data.reset_index()
data = data.sort_values(["ticker","date"]).reset_index(drop=True)
data["rendement_journalier"] = data.groupby("ticker")["close"].pct_change(fill_method=None) * 100
data.set_index(["date","ticker"],inplace=True)

data["variation_intrajournaliere"] = (
    (data["close"] - data["open"])
    / data["open"]
    * 100
)

data["amplitude_journaliere"] = (
    (data["high"] - data["low"])
    / data["open"]
    * 100
)

data["variation_volume"] = (
    data.groupby("ticker")["volume"]
    .pct_change(fill_method=None)
    * 100
)

data["direction"] = np.select(
    [
        data["rendement_journalier"] > 0,
        data["rendement_journalier"] < 0
    ],
    [
        1,
        -1
    ],
    default=0
)
data.head(10)

In [ ]:
from pathlib import Path
output_dir = Path("../data/preprocessed_data")
output_dir.mkdir(parents=True, exist_ok=True)
data.to_csv(output_dir / "stock_data.csv", index=False)